# 🎓 教师面板 | Teacher Dashboard

成绩查看 · 统计分析 · 可视化 · 学生管理（名册）

> 使用前：管理员需在主机端运行 `collect_grades.py` 汇总学生成绩（或已设定时任务）。本面板读取 `~/grades/all_results.json`。


In [ ]:
import sys, os, json
sys.path.insert(0, '/home/jovyan/labs')
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML, Image
from common import gradebook
from common import student_admin as sa
from common import skills_profiler
from common import chain_validator
from common import report_checker

from common import plagiarism_detector
from common import semester_archiver
COLLECTED = '/home/jovyan/grades/all_results.json'
print('面板就绪。数据文件:', '存在' if os.path.exists(COLLECTED) else '未找到（先运行汇总脚本，仅可做学生管理）')


## 一、查看成绩 | View Grades

点击按钮，列出全班每生每实验得分与平均分。


In [ ]:
out_grades = widgets.Output()
btn_grades = widgets.Button(description="📋 查看成绩", button_style="primary")

def on_grades(_):
    with out_grades:
        clear_output()
        if not os.path.exists(COLLECTED):
            print("暂无汇总数据。"); return
        table = gradebook.build_grade_table(gradebook.load_collected(COLLECTED))
        if not table:
            print("暂无学生成绩。"); return
        print(f"{'学生':<16}{'平时完成':>8}{'完成率':>8}{'平时成绩':>10}{'已做均分':>10}{'Capstone':>10}{'参考总评':>10}")
        print("-" * 76)
        for user, row in sorted(table.items()):
            n_str = f"{row.get('n_done', 0)}/{row.get('n_required', 29)}"
            rate_str = f"{row.get('completion_rate', 0.0):.1f}%"
            cap_val = row.get('capstone_score')
            cap_str = f"{cap_val:.0f}分" if cap_val is not None else "未完成"
            comp_str = f"{row.get('course_composite', 0.0):.1f}"
            print(f"{user:<16}{n_str:>8}{rate_str:>8}{row.get('total', 0.0):>10.1f}{row.get('submitted_avg', 0.0):>10.1f}{cap_str:>10}{comp_str:>10}")
        print()
        print("💡 说明：平时成绩以全量 29 门专业实验为基准（缺交按 0 分计入）；Capstone 为期末 25% 独立大作业；参考总评按大纲规则（平时60% + Capstone25% + 期中测验锚点15%）折算。")
btn_grades.on_click(on_grades)
display(btn_grades, out_grades)


## 二、成绩统计与可视化 | Statistics & Charts

全班均分/中位数/及格率 + 三张图：分数分布、每实验均分、学生排名。


In [ ]:
out_stats = widgets.Output()
btn_stats = widgets.Button(description="📊 成绩统计与图表", button_style="success")

def on_stats(_):
    with out_stats:
        clear_output()
        if not os.path.exists(COLLECTED):
            print("暂无汇总数据。"); return
        table = gradebook.build_grade_table(gradebook.load_collected(COLLECTED))
        st = gradebook.statistics(table)
        print(f"学生总数: {st.get('n_students', 0)}  提交人数: {st.get('n_submitted', 0)}  "
              f"全班总评均分: {st.get('mean', 0.0)}  已做实验均分: {st.get('submitted_mean', 0.0)}  "
              f"平均完成率: {st.get('mean_completion_rate', 0.0)}%  及格率: {st.get('pass_rate', 0.0)}%")
        print()
        print("各实验（均分 / 提交人数 / 及格率）：")
        for exp, v in sorted(st.get('per_experiment', {}).items()):
            print(f"  {exp:<28} {v['mean']:>5} 分  n={v['n']:<3}  及格率 {v['pass_rate']}%")
        from IPython.display import Image
        for fn in [gradebook.chart_distribution(table),
                   gradebook.chart_per_experiment(st),
                   gradebook.chart_ranking(table)]:
            if fn: display(Image(filename=fn))
btn_stats.on_click(on_stats)
display(btn_stats, out_stats)


## 三、7 大核心工程能力雷达与全班短板画像 | Skills Profiler

将全课程 29 门实验映射至 7 大核心工程能力维度，全班能力短板热力诊断与单生能力雷达画像。

In [ ]:
out_skills = widgets.Output()
btn_class_skills = widgets.Button(description="🎯 全班能力短板诊断", button_style="info")
dropdown_student = widgets.Dropdown(description="单生雷达图:", options=["-- 请先点击诊断 --"])

_global_class_table = {}

def on_class_skills(_):
    global _global_class_table
    with out_skills:
        clear_output()
        if not os.path.exists(COLLECTED):
            print("暂无汇总数据。"); return
        raw_data = gradebook.load_collected(COLLECTED)
        _global_class_table = gradebook.build_grade_table(raw_data)
        if not _global_class_table:
            print("暂无学生成绩。"); return
        class_matrix = skills_profiler.compute_class_skills_matrix(_global_class_table)
        display(HTML(skills_profiler.render_class_weakness_html(class_matrix)))
        students = [u for u in sorted(_global_class_table.keys()) if u not in ("admin", "lost+found")]
        dropdown_student.options = students
        if students:
            dropdown_student.value = students[0]
            p = skills_profiler.compute_student_skill_profile(_global_class_table[students[0]])
            display(HTML(skills_profiler.render_student_radar_svg(p, students[0])))

def on_student_change(change):
    if change["type"] == "change" and change["name"] == "value":
        stu = change["new"]
        if stu in _global_class_table:
            with out_skills:
                clear_output()
                class_matrix = skills_profiler.compute_class_skills_matrix(_global_class_table)
                display(HTML(skills_profiler.render_class_weakness_html(class_matrix)))
                p = skills_profiler.compute_student_skill_profile(_global_class_table[stu])
                display(HTML(skills_profiler.render_student_radar_svg(p, stu)))

btn_class_skills.on_click(on_class_skills)
dropdown_student.observe(on_student_change)
display(widgets.HBox([btn_class_skills, dropdown_student]), out_skills)


## 四、跨实验数据链路关联分析 (D3 电池标定 → C1 整车能耗) | Data Chain

分析全班“D3 电池 OCV-SOC 标定最大误差”与“C1 UDDS 百公里电耗仿真方差”的工程敏感性散点回归。

In [ ]:
out_chain = widgets.Output()
btn_chain = widgets.Button(description="🔗 电池标定 vs 能耗关联分析", button_style="warning")

def on_chain(_):
    with out_chain:
        clear_output()
        if not os.path.exists(COLLECTED):
            print("暂无汇总数据。"); return
        raw_data = gradebook.load_collected(COLLECTED)
        analysis = chain_validator.analyze_battery_energy_chain(raw_data)
        print(f"有效数据样本数: {analysis['n_samples']} | Pearson 相关系数 r = {analysis['correlation_r']:.3f}")
        print(f"全班 D3 平均标定误差: {analysis['mean_cal_err']:.2f}% | C1 平均百公里电耗: {analysis['mean_energy']:.2f} kWh/100km")
        png_path = chain_validator.plot_chain_correlation(raw_data)
        if png_path and os.path.exists(png_path):
            display(Image(filename=png_path))
        else:
            print("样本不足 2 人，待更多学生完成 D3 与 C1 后生成关联图。")

btn_chain.on_click(on_chain)
display(btn_chain, out_chain)


## 五、实验报告数字一致性批量核验 (60% 权重防伪) | Report Checker

对学生提交的 Markdown 报告数值与实际仿真运行归档 JSON 进行规则核验，筛查数据造假或抄袭。

In [ ]:
out_report = widgets.Output()
btn_report = widgets.Button(description="🛡️ 实验报告一致性核验", button_style="danger")

def on_report(_):
    with out_report:
        clear_output()
        print("正在扫描全班实验报告 Markdown 与结果归档...")
        class_summaries = report_checker.check_class_reports("/opt/ievlab/home")
        if not class_summaries:
            print("当前未检测到已提交的独立实验报告 Markdown 文件。")
            print("提示：学生可在其 home 或 notebooks 目录编写 <exp_id>_report.md。")
            return
        print(f"检测到 {len(class_summaries)} 位学生提交了报告：")
        for user, exp_sums in class_summaries.items():
            print()
            print(f"👤 学生: {user}")
            for exp_id, s in exp_sums.items():
                c_score = s.get("consistency_score", 0.0)
                m_cnt = s.get("matched_count", 0)
                t_cnt = s.get("total_evaluated", 0)
                verdict = s.get("verdict", "")
                badge = "🟢 PASS" if c_score >= 80 and not s.get("severe_discrepancy") else "🔴 ALERT"
                print(f"  [{badge}] {exp_id}: 一致性得分 {c_score:.1f} ({m_cnt}/{t_cnt} 一致) -> {verdict}")

btn_report.on_click(on_report)
display(btn_report, out_report)


## 六、学生管理（名册）| Student Roster

平台账号由学生用学校邮箱自助注册；此处管理**班级名册**（学号/班级/状态）。

In [ ]:
# 添加单个学生
w_email = widgets.Text(description='邮箱前缀', placeholder='如 zhangsan')
w_name = widgets.Text(description='姓名')
w_sid = widgets.Text(description='学号')
w_class = widgets.Text(description='班级')
out_add = widgets.Output()
btn_add = widgets.Button(description='➕ 添加学生', button_style='info')

def on_add(_):
    with out_add:
        clear_output()
        if not w_email.value.strip():
            print('请填邮箱前缀'); return
        s = sa.add_student(w_email.value, w_name.value, w_sid.value, w_class.value)
        print('已添加/已存在:', s['username'], s['email'])
btn_add.on_click(on_add)
display(widgets.VBox([w_email, w_name, w_sid, w_class, btn_add, out_add]))


In [ ]:
# 批量导入CSV（列：email,name,student_id,class_name，可用中文表头：邮箱,姓名,学号,班级）
w_csv = widgets.Textarea(description='CSV粘贴', layout=widgets.Layout(width='90%', height='120px'),
                         placeholder='email,name,student_id,class_name\nzhangsan@zhjpec.edu.cn,张三,20230101,汽修2301')
out_imp = widgets.Output()
btn_imp = widgets.Button(description='📥 批量导入', button_style='info')

def on_imp(_):
    with out_imp:
        clear_output()
        ok, skip, errs = sa.import_students_csv(w_csv.value)
        print(f'导入成功 {ok} 人，跳过（重复）{skip} 人')
        for e in errs: print('⚠️', e)
btn_imp.on_click(on_imp)
display(widgets.VBox([w_csv, btn_imp, out_imp]))


In [ ]:
# 名册查看与归档（已结课）
out_list = widgets.Output()
btn_list = widgets.Button(description='👥 查看名册', button_style='primary')
w_arch = widgets.Text(description='归档用户名', placeholder='输入username后点归档')
btn_arch = widgets.Button(description='🗄️ 归档（已结课）', button_style='warning')
out_arch = widgets.Output()

def on_list(_):
    with out_list:
        clear_output()
        st = sa.roster_stats()
        print(f"总人数 {st['total']}（在读 {st['active']} / 已归档 {st['archived']}）班级分布: {st['classes']}")
        for s in sa.list_students():
            print(f"  {s['username']:<16}{s.get('name',''):<8}{s.get('student_id',''):<12}{s.get('class_name',''):<12}{s['status']}")
def on_arch(_):
    with out_arch:
        clear_output()
        s = sa.archive_student(w_arch.value.strip())
        print('已归档:', s['username'] if s else '未找到该学生')
btn_list.on_click(on_list)
btn_arch.on_click(on_arch)
display(widgets.VBox([btn_list, out_list, w_arch, btn_arch, out_arch]))


## 七、学生代码 AST 语法树查重引擎 | Code Plagiarism Detector

基于抽象语法树 (AST) 结构化清洗，忽略变量更名、注释增删与格式伪装，精准识别跨学生算法抄袭。

In [ ]:
out_plag = widgets.Output()
w_exp_plag = widgets.Dropdown(
    description="选择实验:",
    options=["A1_cutin_aeb", "A3_aeb_closedloop", "B1_idm_calibration", "E3_mpc_follow", "E4_dp_ems", "Capstone_iev_integrated"],
    value="A1_cutin_aeb"
)
btn_plag = widgets.Button(description="🛡️ 扫描全班代码抄袭", button_style="warning")

def on_plag_scan(_):
    with out_plag:
        clear_output()
        exp = w_exp_plag.value
        print(f"正在扫描全班学生目录下的 {exp} 代码单元格并构建 AST 语法树...")
        rep = plagiarism_detector.scan_class_code_plagiarism("/opt/ievlab/home", exp)
        display(HTML(plagiarism_detector.render_plagiarism_report_html(rep)))

btn_plag.on_click(on_plag_scan)
display(widgets.HBox([w_exp_plag, btn_plag]), out_plag)


## 八、学期结课总归档与数据包导出 | Semester Final Archive

一键生成全班 16 周最终成绩大表 (CSV)、7 大工程能力画像矩阵 (JSON) 以及学术诚信审计总报告并打包为 ZIP：

In [ ]:
out_archive = widgets.Output()
btn_archive = widgets.Button(description="📦 导出学期结课总归档 (ZIP)", button_style="success")

def on_export_archive(_):
    with out_archive:
        clear_output()
        print("正在聚合全班成绩、能力矩阵与诚信审计记录...")
        res = semester_archiver.export_teacher_semester_archive()
        if res.get("success"):
            zip_p = res.get("zip_path", "")
            print(f"✓ 学期总归档打包完成: {zip_p}")
            display(HTML(semester_archiver.render_download_link_html(zip_p, "📥 一键下载全班学期总归档 (ZIP)")))
        else:
            print("导出失败:", res.get("error"))

btn_archive.on_click(on_export_archive)
display(btn_archive, out_archive)


## 九、Phase 0 课前预习打卡与微任务心得看板 | Novice Preview Cockpit

查看全班学生在 30 门实验中的零代码直觉探索打卡记录、调节的关键参数与填写的思考心得。


In [ ]:
out_prev = widgets.Output()
btn_prev = widgets.Button(description="🌟 查阅课前预习打卡", button_style="warning")

def on_view_previews(_):
    with out_prev:
        clear_output()
        prev_path = "/home/jovyan/grades/all_previews.json"
        if not os.path.isfile(prev_path):
            # 容错尝试其他路径
            for alt in ["/opt/ievlab/home/teacher/grades/all_previews.json", os.path.expanduser("~/grades/all_previews.json")]:
                if os.path.isfile(alt):
                    prev_path = alt
                    break

        if not os.path.isfile(prev_path):
            print("当前暂无预习打卡汇总数据 (all_previews.json)。")
            print("提示：学生在 Notebook 的 Phase 0 中点击“预习打卡保存”后，系统将在每小时自动巡检汇聚。")
            return
        try:
            with open(prev_path, "r", encoding="utf-8") as f:
                all_prev = json.load(f)
        except Exception as e:
            print(f"读取打卡汇总失败: {e}")
            return

        if not all_prev:
            print("目前还没有学生提交课前预习打卡记录。")
            return

        total_checkins = sum(len(exps) for exps in all_prev.values())
        print(f"📊 课前打卡学情概况：共 {len(all_prev)} 位学生参与，累计打卡 {total_checkins} 次。")
        print("=" * 65)

        for u, exps in all_prev.items():
            print(f"\n👤 学生: {u} (已预习 {len(exps)} 门实验)")
            for exp_id, pdata in exps.items():
                insight = pdata.get("student_insight", "（无文字心得）")
                params = [f"{k}={v}" for k, v in pdata.items() if k not in ("experiment", "mode", "student_insight")]
                param_str = ", ".join(params)
                print(f"  • [{exp_id}] 参数: ({param_str})")
                print(f"    💬 导学心得: “{insight}”")

btn_prev.on_click(on_view_previews)
display(btn_prev, out_prev)


---

### 说明 | Notes
- 成绩判据来自 `common/evaluator.py`（阈值=真实引擎实测值），60分及格。
- 归档仅改名册状态，保留学生信息与成绩关联，随时可恢复（`unarchive_student`）。
- 汇总脚本：主机端 `/opt/ievlab/deploy/collect_grades.py`，建议cron每小时运行。
